In [ ]:
import itertools
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import mean_squared_error
from core.derivatives.neural import NeuralDerivatives

def find_optimal_nn_params(df, col_name, time_col, analytic_obj, comp_idx, 
                           epochs_grid=[2000, 4000], 
                           lr_grid=[0.001, 0.005],
                           hidden_grid=[64, 128]):
    """
    Ищет оптимальные параметры нейросети, минимизируя ошибку производных.
    """
    # Подготовка данных
    df_sorted = df.sort_values(by=time_col)
    t = df_sorted[time_col].values
    y = df_sorted[col_name].values
    
    # Истинные производные
    d1_true_all, d2_true_all, d3_true_all = analytic_obj.get_derivatives(t)
    true_d1 = d1_true_all[comp_idx]
    true_d2 = d2_true_all[comp_idx]
    true_d3 = d3_true_all[comp_idx]
    
    # Масштабы для нормировки ошибки
    sc1 = np.max(np.abs(true_d1))**2 if np.max(np.abs(true_d1)) > 0 else 1.0
    sc2 = np.max(np.abs(true_d2))**2 if np.max(np.abs(true_d2)) > 0 else 1.0
    sc3 = np.max(np.abs(true_d3))**2 if np.max(np.abs(true_d3)) > 0 else 1.0

    results = []
    
    # Перебор всех комбинаций
    for ep, lr, hd in itertools.product(epochs_grid, lr_grid, hidden_grid):
        print(f"Testing: Epochs={ep}, LR={lr}, Hidden={hd}...", end="\r")
        
        # Фиксируем seed для воспроизводимости
        torch.manual_seed(42)
        np.random.seed(42)
        
        # Создаем и обучаем модель
        nn_deriv = NeuralDerivatives(hidden_dim=hd, lr=lr, epochs=ep, device='cpu')
        
        try:
            nn_deriv.fit(t, y)
            
            # Считаем производные
            d1, d2, d3 = nn_deriv.compute_derivatives(t)
            
            # Считаем ошибки (без краев)
            sl = slice(5, -5)
            
            mse1 = mean_squared_error(true_d1[sl], d1[sl])
            mse2 = mean_squared_error(true_d2[sl], d2[sl])
            mse3 = mean_squared_error(true_d3[sl], d3[sl])
            
            # Взвешенная оценка (приоритет 3-й производной)
            score = 1.0*(mse1/sc1) + 1.0*(mse2/sc2) + 5.0*(mse3/sc3)
            
            results.append({
                'epochs': ep,
                'lr': lr,
                'hidden': hd,
                'score': score,
                'mse1': mse1,
                'mse3': mse3
            })
            
        except Exception as e:
            print(f"Error with params {ep, lr, hd}: {e}")
            
    print("\nSearch complete.")
    
    res_df = pd.DataFrame(results)
    best_idx = res_df['score'].idxmin()
    best_params = res_df.loc[best_idx].to_dict()
    
    return best_params, res_df

In [ ]:
# 1. Параметры для перебора
# Можно задать шире, но для демонстрации хватит небольшого
epochs_options = [2000, 4000, 6000]
lr_options = [0.001, 0.005, 0.01]
hidden_options = [64, 128]

# 2. Запуск поиска (на основе аналитики для Eps_3)
best_nn_params, res_nn_df = find_optimal_nn_params(
    df_stage2,
    col_name='Eps_3',
    time_col='Time_Local',
    analytic_obj=analytical, # Используем идеальную аналитику (nu=0.5) или физичную (nu=0.29) - для Eps_3 неважно
    comp_idx=2,
    epochs_grid=epochs_options,
    lr_grid=lr_options,
    hidden_grid=hidden_options
)

print("\nЛУЧШИЕ ПАРАМЕТРЫ НЕЙРОСЕТИ:")
print(f"Epochs: {int(best_nn_params['epochs'])}")
print(f"LR:     {best_nn_params['lr']}")
print(f"Hidden: {int(best_nn_params['hidden'])}")

In [ ]:
# Извлекаем лучшие параметры
best_epochs = int(best_nn_params['epochs'])
best_lr = best_nn_params['lr']
best_hidden = int(best_nn_params['hidden'])

components = ['Eps_1', 'Eps_2', 'Eps_3']

print("Обучение финальных моделей...")

for col in components:
    print(f"Processing {col}...")
    
    # 1. Инициализация с лучшими параметрами
    # Важно: Seed внутри класса не фиксируется жестко, 
    # можно добавить torch.manual_seed(42) перед каждым созданием для полной повторяемости
    torch.manual_seed(42) 
    
    nn_model = NeuralDerivatives(
        hidden_dim=best_hidden, 
        lr=best_lr, 
        epochs=best_epochs, 
        device='cpu'
    )
    
    # 2. Обучение на 100% данных
    t_vals = df_stage2['Time_Local'].values
    y_vals = df_stage2[col].values
    
    nn_model.fit(t_vals, y_vals)
    
    # 3. Расчет производных
    d1, d2, d3 = nn_model.compute_derivatives(t_vals)
    
    # 4. Сохранение в DataFrame
    # Суффикс _nn
    df_stage2[f'd{col}_nn'] = d1
    df_stage2[f'd2{col}_nn'] = d2
    df_stage2[f'd3{col}_nn'] = d3

print("Готово!")

In [ ]:
from core.calculations import calculate_geometry

# 1. Расчет геометрии для NN
s_nn, k_nn, t_nn = calculate_geometry(df_stage2, suffix='_nn')

df_stage2['kappa_nn'] = k_nn
df_stage2['tau_nn'] = np.abs(t_nn) # Берем модуль

# 2. Визуализация: БИТВА МЕТОДОВ
# Кручение - самый сложный параметр

plot_xy(
    data_pairs=[(model_name, df_stage2)],
    x_col="Time_Local",
    y_cols=["tau_nn", "tau_analytic"],
)

plot_xy(
    data_pairs=[(model_name, df_stage2)],
    x_col="Time_Local",
    y_cols=["kappa_nn", "kappa_analytic"],
)

In [ ]:
# plot_xy(
#     data_pairs=[(model_name, df_stage2)],
#     x_col="Time_Local",
#     y_cols=["dEps_1_nn", "dEps_1_analytic"],
# )
# plot_xy(
#     data_pairs=[(model_name, df_stage2)],
#     x_col="Time_Local",
#     y_cols=["d2Eps_1_nn", "d2Eps_1_analytic"],
# )
# plot_xy(
#     data_pairs=[(model_name, df_stage2)],
#     x_col="Time_Local",
#     y_cols=["d3Eps_3_nn", "d3Eps_3_analytic"],
# )